# Complexity-03 — Plus de temps, plus de problèmes

**Navigation** : [Index](README.md) | [02 — Vérifier ou trouver](Complexity-02-P-NP-Reduction.ipynb) | [03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb) | [03c — Zoo navigable](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb)

> **Public : Licence.** La démonstration historique et sa simulation multitête sont conservées dans l'approfondissement 03b.

## Ce que ce notebook suppose

- [01 — Compter des pas](Complexity-01-StepCounting.ipynb) : taille et budget de pas ;
- [02 — Vérifier ou trouver](Complexity-02-P-NP-Reduction.ipynb) : problème de décision et algorithme.

**Objectifs** : distinguer inclusion et séparation stricte de classes, comprendre une diagonale finie et expliquer pourquoi une table n'est pas une preuve de hiérarchie. **Durée estimée** : 35 minutes.

**Le théorème en une phrase.** Le théorème de hiérarchie temporelle (Hartmanis–Stearns, 1965) affirme que, pour des budgets raisonnables, disposer de strictement plus de temps permet de décider strictement plus de problèmes : certaines tâches exigent réellement plus de pas, et aucun algorithme rapide ne peut les simuler. Ce carnet illustre les **mécanismes** de cette preuve — budgets imbriqués, diagonalisation, simulation bornée — sur des tables finies ; il ne prétend pas la démontrer, ce que rappelle chaque lecture.

## 1. Des budgets imbriqués

Si un programme répond en n pas, il répond aussi dans un budget de n² pas. Cela montre une **inclusion** des budgets, pas une séparation des classes de problèmes : deux programmes différents sur un même problème ne prouvent pas qu'aucun programme rapide ne peut le résoudre. Comparons les budgets sur cinq tailles.

L'inclusion est triviale : un programme qui s'arrête en n pas s'arrête aussi avant n² pas. La difficulté est ailleurs — montrer qu'un problème exige *plus* que n² pas demande d'écarter **tous** les programmes rapides, pas seulement celui qu'on a sous la main. C'est ce fossé entre « j'ai un exemple » et « j'ai épuisé les possibilités » que ce carnet creuse.

In [1]:
tailles = (2, 4, 8, 16, 32)
print(f"{'n':>3} | {'n²':>5} | {'2ⁿ':>12}")
for n in tailles:
    print(f"{n:>3} | {n * n:>5} | {2 ** n:>12}")
assert all(n * n <= 2 ** n for n in tailles)


  n |    n² |           2ⁿ
  2 |     4 |            4
  4 |    16 |           16
  8 |    64 |          256
 16 |   256 |        65536
 32 |  1024 |   4294967296


**Lecture du résultat.** Les coûts s'ordonnent pour les tailles affichées : à n = 32, le budget n² vaut 1 024 pas contre 4 294 967 296 pour 2ⁿ — l'écart se creuse à chaque taille. Pour conclure qu'une classe possède un problème absent de l'autre, il faudrait écarter **tous** les algorithmes au petit budget. La table numérique ne le fait pas.

## 2. L'idée de diagonalisation

Énumérons quelques machines candidates. À la ligne i, la construction D choisit l'opposé de la réponse de la machine numéro i sur l'entrée i. D diffère donc de chaque machine listée sur au moins une entrée. La liste ci-dessous n'est qu'une expérience **finie** : elle ne représente pas toutes les machines possibles.

La diagonalisation inverse le sens de la question : au lieu de chercher un algorithme rapide, on construit un problème D qui contredit chaque machine candidate sur son propre terrain — l'entrée i. Contre une liste finie, cela marche ; contre l'infinité des machines, il faudrait une construction et une borne universelles, ce que la section 4 discute.

In [2]:
machines = (
    lambda i: i % 2 == 0,
    lambda i: i < 2,
    lambda i: True,
    lambda i: i % 3 == 0,
)
diagonale = {i: not machine(i) for i, machine in enumerate(machines)}
print(f"{'i':>2} | {'M_i(i)':>6} | {'D(i)':>5} | différents")
for i, machine in enumerate(machines):
    print(f"{i:>2} | {str(machine(i)):>6} | {str(diagonale[i]):>5} | {diagonale[i] != machine(i)}")
assert all(diagonale[i] != machine(i) for i, machine in enumerate(machines))


 i | M_i(i) |  D(i) | différents
 0 |   True | False | True
 1 |   True | False | True
 2 |   True | False | True
 3 |   True | False | True


**Lecture du résultat.** Les quatre lignes affichent `True` dans la colonne « différents » : D(i) vaut l'opposé de M_i(i) sur chacune des entrées listées (0, 1, 2 et 3). Il reste une infinité de machines non listées ; ce tableau ne prouve donc aucune séparation de classes.

## 3. Le budget de simulation

Une machine qui ne répond pas avant la limite ne peut pas bloquer indéfiniment le constructeur diagonal. Ici les coûts sont **déclarés** par un modèle fini : ce ne sont pas des pas mesurés par un simulateur universel. Une réponse non observée n'est pas un verdict faux.

Dans la preuve réelle, cette limite est gérée par une **horloge universelle** : on simule chaque machine candidate pas à pas et on la stoppe au-delà du budget. Payer à la fois la machine et son simulateur est précisément le surcoût qui fait tenir la hiérarchie — et que le modèle fini ici ne rend pas.

In [3]:
candidates = ((True, 1), (False, 3), (True, 12), (False, 40))
def budget(i):
    return 2 * i + 4

print(f"{'i':>2} | {'pas':>3} | {'budget':>6} | {'observé':>7} | {'D(i)':>5}")
for i, (verdict, pas) in enumerate(candidates):
    observe = verdict if pas <= budget(i) else None
    diagonal = not observe if observe is not None else True
    print(f"{i:>2} | {pas:>3} | {budget(i):>6} | {str(observe):>7} | {str(diagonal):>5}")


 i | pas | budget | observé |  D(i)
 0 |   1 |      4 |    True | False
 1 |   3 |      6 |   False |  True
 2 |  12 |      8 |    None |  True
 3 |  40 |     10 |    None |  True


**Lecture du résultat.** `None` signifie « non observé dans le budget », jamais « faux » : aux lignes 2 et 3, les coûts déclarés (12 et 40) dépassent le budget (8 et 10), la simulation s'interrompt et la réponse n'est pas observée. Une machine dépassant ce budget sur une entrée ne satisfait pas la borne uniforme sur **toutes** les entrées, mais le tableau ne caractérise aucune classe entière.

## 4. Le théorème, sans raccourci expérimental

Le théorème de hiérarchie temporelle établit, sous des hypothèses techniques, qu'il existe des problèmes décidables avec un budget suffisamment plus grand mais impossibles avec le plus petit. Il faut énumérer toutes les machines admissibles, construire une horloge et payer le coût de simulation universelle. Les tables finies n'accomplissent aucune de ces preuves. [03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb) détaille le modèle historique. Ce résultat ne résout pas P contre NP.

Concrètement, la séparation s'obtient en construisant un problème D qui simule chaque machine candidate M_i sur son entrée i, répond l'opposé, et le fait dans un budget à peine supérieur à celui de M_i — d'où une classe strictement plus grande quand le budget croît. Le prix est connu : énumérer toutes les machines, les doter d'une horloge universelle, et simuler chacune en payant la machine et son simulateur.

## Exercices

Les trois exercices prolongent les démonstrations sur les mêmes objets finis. Ils ne demandent pas de prouver la hiérarchie — hors de portée d'une table — mais de **sentir** ce que chaque table montre et ce qu'elle laisse ouvert, en manipulant budgets, diagonale et borne de simulation.

### Exercice 1 — Inclusion ou séparation ?

Comparez n² et n³ pour n = 2, 4, 8, 16. Expliquez pourquoi ces nombres ne prouvent pas qu'un problème devient soluble uniquement avec le second budget.

- **Indice :** une table de nombres n'écarte aucun autre algorithme.
- **Etape 1 :** produire les quatre couples.
- **Etape 2 :** distinguer inclusion et séparation.

In [4]:
comparaison_budgets = None  # TODO etudiant : couples (n², n³)
print('Exercice a completer : inclusion et separation')


Exercice a completer : inclusion et separation


### Exercice 2 — Une autre diagonale

Ajoutez une cinquième fonction à `machines`, puis reconstruisez D et expliquez pourquoi cette table finie n'est toujours pas une preuve générale.

- **Indice :** évaluer la nouvelle fonction sur son propre indice.
- **Etape 1 :** ajouter une machine.
- **Etape 2 :** recalculer les cinq oppositions.

In [5]:
nouvelle_diagonale = None  # TODO etudiant : cinquieme machine et nouvelle diagonale
print('Exercice a completer : nouvelle diagonale')


Exercice a completer : nouvelle diagonale


### Exercice 3 — Dépassement de budget

Repérez les lignes dont le coût déclaré dépasse `budget(i)`. Proposez un budget qui couvre les quatre coûts et expliquez pourquoi cela ne construit pas une borne pour toutes les machines.

- **Indice :** comparer les deux colonnes numériques.
- **Etape 1 :** produire un nouveau budget.
- **Etape 2 :** énoncer la limite de ce modèle fini.

In [6]:
budget_etendu = None  # TODO etudiant : couvrir les quatre couts
print('Exercice a completer : budget etendu')


Exercice a completer : budget etendu


## Conclusion

| Geste | Vérifié ici | Non démontré ici |
|---|---|---|
| Budgets | Inclusion numérique | Séparation stricte de classes |
| Diagonale | Quatre contradictions finies | Toutes les machines admissibles |
| Simulation | Arrêt sur dépassement | Horloge et coût universel bornés |

**À retenir.** Trois gestes, trois portées : l'inclusion des budgets se mesure, la diagonalisation se teste sur une liste finie, la simulation bornée s'observe sur un dépassement — mais aucune table finie n'énumère toutes les machines ni ne construit l'horloge universelle. La hiérarchie temporelle est un théorème sur l'infini des algorithmes ; ce carnet en montre les mécanismes sur le fini, pour que la preuve de 03b ait un terrain.

Le théorème affirme l'existence de problèmes demandant réellement plus de temps, et non seulement de programmes lents. L'approfondissement [03b](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb) travaille le résultat historique.

**Référence** : Hartmanis, J. et Stearns, R. E. (1965), *On the computational complexity of algorithms*, Transactions of the American Mathematical Society 117, 285–306.

## Relativisation : ce qu'une diagonale finie ne peut pas trancher

Les trois tableaux de ce carnet — budgets imbriqués, diagonale finie, simulation bornée — illustrent les **mécanismes** d'une preuve de séparation. Le théorème réel (Hartmanis–Stearns 1965) est inconditionnel : il sépare des classes de problèmes, pas seulement des programmes. Mais la technique qui le porte — la diagonalisation — a une limite connue depuis 1975, et cette limite mérite sa place ici parce qu'elle dit exactement ce que ce type de geste peut et ne peut pas trancher.

**Le théorème des oracles (Baker–Gill–Solovay 1975).** Il existe un oracle récursif A tel que `P^A = NP^A`, et un oracle récursif (sparse) B tel que `P^B ≠ NP^B`. Un oracle est une boîte noire gratuite ajoutée aux deux côtés de l'équation ; le résultat fabriqué par BGS montre que la question « P = NP ? » a les **deux** réponses selon l'oracle. Conséquence immédiate : toute technique de preuve qui *relativise* — qui s'applique tel quel en présence d'un oracle arbitraire — est structurellement neutre sur P vs NP, dans les deux sens. Les auteurs l'écrivent dès leur introduction : la diagonalisation ordinaire est du nombre des techniques que l'on attend relativisantes, et l'on ne voit pas non plus de méthode générale de simulation qui échapperait à l'objection. (Baker, Gill, Solovay, *Relativizations of the P NP question*, SIAM Journal on Computing 4(4), 1975, p. 431 — copie archivée au gisement bibliographique du cluster.)

**Ce que cela change pour la diagonale de ce carnet.** Ajoutez mentalement une colonne « oracle » au tableau de la section 2 : la contradiction finie (machines ≠ au point diagonal) survit inchangée. C'est la marque d'un argument qui relativise — il ne peut donc rien conclure sur P vs NP, seulement sur les objets finis qu'il exhibe. Ce n'est pas un défaut du carnet : la séparation de Hartmanis–Stearns, elle, est démontrée ; la leçon est que le geste diagonal enseigné ici est le bon outil pour les hiérarchies de temps, et le mauvais pour la question P vs NP.

**Une nuance de mesure.** Mehlhorn (rapporté §3 de BGS) : les oracles X avec `P^X = NP^X` forment un ensemble *maigre* — en un sens précis, « la plupart » des oracles séparent. L'intuition catégorielle penche donc séparation, et pourtant la question reste ouverte : la relativisation n'est pas un indice de réponse, c'est la preuve que les méthodes standard n'en fournissent pas.

**Écho contemporain.** Le doute exprimé par Wolfram en 2026 — résoudre P vs NP par une preuve finie depuis les axiomes standard, « the results here make me doubt it » — est la face empirique de la même prudence : là où BGS démontre un théorème *sur les techniques de preuve*, le recensement empirique des machines de Turing petites n'offre qu'une attente. Les deux se citent l'un l'autre avec leur statut respectif : prouvé (1975) et attendu (2026).